# Step 4 & 6: 1D ResNet Architecture + Training with Adam

Train the full 1D ResNet (~790K params) with Adam optimizer.

Hyperparameters (paper defaults):
- α=0.001, β1=0.9, β2=0.999, ε=1e-8
- Batch=128, Max epochs=100, Early stopping patience=15
- CosineAnnealingLR, Dropout=0.4, weight_decay=1e-4
- FocalLoss (γ=2, α_t=1/class_freq)

Target: ≥98% accuracy, ≥93% macro F1

In [5]:
import sys, os
from pathlib import Path
_r = Path.cwd()
while not (_r / 'src').is_dir():
    _r = _r.parent
os.chdir(_r)
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))

import torch
from src.utils.seed import set_seed
from src.utils.device import get_device
from src.models.resnet1d import ResNet1D
from src.training.focal_loss import FocalLoss, compute_class_frequencies
from src.training.trainer import Trainer
from src.training.callbacks import EarlyStopping, ModelCheckpoint
from src.optimizers.optimizer_factory import build_optimizer

In [6]:
set_seed(42)
device = get_device()
model = ResNet1D(num_classes=5, dropout=0.4).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Parameters: 790,181


In [7]:
from src.data.loader import load_mitbih
from src.data.preprocessor import preprocess
from src.data.dataset import build_dataloaders

X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_mitbih()
splits = preprocess(X_train_raw, y_train_raw, X_test_raw, y_test_raw, val_fraction=0.15)
train_loader, val_loader, test_loader = build_dataloaders(splits, batch_size=128)
print(f'Data ready — train: {splits["X_train"].shape}, val: {splits["X_val"].shape}, test: {splits["X_test"].shape}')

Data ready — train: (307870, 187, 1), val: (13134, 187, 1), test: (21892, 187, 1)


In [8]:
optimizer = build_optimizer(model, 'adam', weight_decay=1e-4)
# Use pre-SMOTE y_train_raw: after SMOTE all classes are ~0.2, giving uniform alpha [1,1,1,1,1]
# which defeats inverse-frequency weighting. Pre-SMOTE frequencies preserve true class imbalance.
class_freqs = compute_class_frequencies(y_train_raw)
focal_loss = FocalLoss(gamma=2.0, class_frequencies=class_freqs)

trainer = Trainer(
    model, optimizer, focal_loss, device,
    callbacks=[EarlyStopping(patience=15), ModelCheckpoint(optimizer_name='adam')],
    use_lr_scheduler=True,
    max_epochs=100,
)
history = trainer.fit(train_loader, val_loader)

Epoch   1/100 | loss 0.0349/0.0546 | f1 0.7472/0.3791 | 18.1s


Epoch   2/100 | loss 0.0141/0.0219 | f1 0.8703/0.5822 | 18.1s


Epoch   3/100 | loss 0.0106/0.0178 | f1 0.9012/0.5935 | 18.2s


Epoch   4/100 | loss 0.0089/0.0202 | f1 0.9161/0.5886 | 18.2s


Epoch   5/100 | loss 0.0076/0.0281 | f1 0.9270/0.5485 | 18.4s


...

Epoch  96/100 | loss 0.0001/0.0214 | f1 0.9989/0.8992 | 19.5s


Epoch  97/100 | loss 0.0001/0.0228 | f1 0.9989/0.9010 | 19.3s


Epoch  98/100 | loss 0.0001/0.0231 | f1 0.9990/0.9026 | 19.6s


Epoch  99/100 | loss 0.0001/0.0233 | f1 0.9991/0.9075 | 19.7s


Epoch 100/100 | loss 0.0001/0.0217 | f1 0.9990/0.8955 | 19.3s
